In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [50]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate

In [39]:
loader = PyPDFLoader("../data/AYUSH_230620.pdf")
docs = loader.load()


In [40]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_data = splitter.split_documents(docs)

In [41]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")

In [45]:
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings,
)

In [46]:
query = "List Down all skils of Ayush Singh Chandel"
data = vector_store.similarity_search(query)

In [47]:
context = ""
for doc in data:
    context += doc.page_content + "\n"
print("Context:", context)

Context: AYUSH SINGH CHANDEL
+91-9693794566⋄Pune, Maharashtra, India
ayushsinghchandel1610@gmail.com⋄linkedin.com/in/ayushsingh⋄github.com/Ayushsingh
EDUCA TION
Army Institute of T echnology , Pune Aug 2023 – Expected May 2027
Bachelor of Engineering, Electronics and Telecommunication Engineering CGPA: 8.17
SKILLS
Languages C/C++, JavaScript, TypeScript, SQL
F rontend React.js, Next.js, T ailwind CSS, Redux, Recoil
Backend Node.js, Express.js, REST APIs, W ebSockets, Serverless
Databases PostgreSQL, MongoDB, Redis, Prisma, Mongoose
DevOps A WS (EC2, S3), Docker, CI/CD, Git, T urborepo
CS F undamentals DSA, OOPs, DBMS, Computer Networks
EXPERIENCE
Smart India Hackathon (Inter-College) Sep 2024 – Oct 2024
•Led a 5-member team to buildIntellectGuard, a unified MERN stack platform centralizing trademark registration,
patent filing, and research publishing into a single ecosystem.
•Spearheaded project planning and UI/UX design, conducted regular agile meetings and creating highly interactiv

In [36]:
llm = ChatOllama(model="gemma3:1b", system_prompt="You are a helpful assistant.")

### Chain - context, prompt, llm, str parser

In [59]:
def get_context(query:str):
    data = vector_store.similarity_search(query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"
    return {
        "context": context,
        "question": query
    }
    

In [60]:
prompt = PromptTemplate.from_template("""
You are a helpful assistant. Answer the question based on the context provided and if the answer is not contained within the context, say 'I don't know'.
Context: {context},
Question: {question}
"""
)

In [62]:
rag_chain = get_context | prompt | llm

In [68]:
res = rag_chain.invoke("What is the age of Ayush Singh Chandel?")

In [69]:
res.content

"I don't know."